# sanity checks for a transient cube

**what is a transient cube?** `transient.tmpl` makes one image per 4 s timestep (about 70 per observation) *after subtracting the static sky model* from the visibilities. stacked together, the images form a cube with axes (time, y, x), so every pixel has a light curve of about 70 points.

* a healthy cube is noise: similar rms at every timestep, mean close to zero, no bright sources left over.
* a transient is a bump in the light curve of one pixel, lasting a few timesteps, with nothing before or after.
* artefacts can look similar: a bad timestep, a source the model subtraction missed, position shifts near bright sources.

**how to use this notebook**

1. edit the settings cell (observation number, folder, options).
2. run all cells. every important figure is saved as a png in `../results_transient_cube_inspect/<obsnum>/`, together with `report.txt`, so you can use them in group discussions.
3. if your files cannot be found, the notebook stops with an error that lists the missing files. it never switches to fake data silently.
4. to see what each check looks like, set `use_demo = True`. this uses a fake cube with problems added on purpose. demo output is clearly marked and saved in a separate `demo` folder.

the thresholds used here (for example 1.5 times the typical rms) are rules of thumb, not pipeline values.

## 0. settings

In [ ]:
# ---- settings you will change ---------------------------------------------------
obsnum = "1476960704"
project_name = "test4_Dev_Birli"
data_dir = "/Users/gechen/setonix_scratch/" + project_name + "/" + obsnum

use_demo = False       # True = ignore your files and use a fake cube with problems added on purpose
check_hdf5 = True      # True = the .hdf5 file must exist and is cross-checked; set False if you have none
cutout = None          # None = whole image, or (y_start, y_end, x_start, x_end) in pixels
box_size = 1000        # width in pixels of the central box used to measure the noise
use_sliders = False    # True = sliders in the single-pixel light curve cell (needs ipywidgets)

# ---- names built from the settings (no need to edit) ----------------------------
file_prefix = obsnum + "_transient"
cube_file = data_dir + "/" + file_prefix + "-cube.fits"
stdev_file = data_dir + "/" + file_prefix + "-stdev.fits"
hdf5_file = data_dir + "/" + file_prefix + ".hdf5"

if use_demo:
    results_dir = "../results_transient_cube_inspect/demo"
else:
    results_dir = "../results_transient_cube_inspect/" + obsnum

## helper functions

small tools used by the later cells. nothing is measured here.

In [ ]:
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.time import Time
from astropy.wcs import WCS
from scipy.ndimage import gaussian_filter1d

warnings.filterwarnings("ignore")        # hide the many harmless astropy / numpy warnings
plt.rcParams["image.origin"] = "lower"   # pixel (0, 0) at the bottom left of every image

problems = []       # things that need a second look, listed in the summary at the end
report_lines = []   # every line printed with say(), saved to report.txt at the end

if use_demo:
    demo_tag = "\n[demo data, not real]"      # second line of every figure title
else:
    demo_tag = ""


def say(text=""):
    """print a line and also remember it for report.txt"""
    print(text)
    report_lines.append(text)


def note_problem(text):
    """record something that needs a look, and print it with a marker in front"""
    say("  !! " + text)
    problems.append(text)


def robust_rms(values):
    """noise estimate that ignores bright sources and nans: 1.4826 times the median absolute deviation"""
    values = np.array(values, dtype=float).flatten()
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan
    median = np.median(values)
    typical_distance_from_median = np.median(np.abs(values - median))
    return 1.4826 * typical_distance_from_median


def make_title(text):
    """figure title that always names the observation, and says so when the data is fake"""
    return obsnum + ": " + text + demo_tag


def save_figure(fig, file_name):
    """save a figure as a png in results_dir, then show it in the notebook"""
    path = os.path.join(results_dir, file_name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    say("saved " + path)
    plt.show()


def make_blob(n_y, n_x, y_centre, x_centre, fwhm):
    """a round gaussian bump with peak value 1 and the given width (fwhm, in pixels), as an image"""
    x_grid, y_grid = np.meshgrid(np.arange(n_x), np.arange(n_y))   # x_grid[j, i] is i, y_grid[j, i] is j
    sigma = fwhm / 2.355
    squared_distance = (x_grid - x_centre) ** 2 + (y_grid - y_centre) ** 2
    return np.exp(-0.5 * squared_distance / sigma ** 2)


def find_top_peaks(image, how_many=5, exclusion_radius=8):
    """find the brightest separate peaks in an image.
    after each peak is found, the area around it is blanked so the next one is a different source."""
    work_image = np.nan_to_num(image)    # nans become 0 (nan_to_num makes a copy, the original is untouched)
    peaks = []
    for count in range(how_many):
        flat_index = np.argmax(work_image)
        y, x = np.unravel_index(flat_index, work_image.shape)
        peaks.append((x, y))
        y_low = max(0, y - exclusion_radius)
        x_low = max(0, x - exclusion_radius)
        work_image[y_low:y + exclusion_radius + 1, x_low:x + exclusion_radius + 1] = 0
    return peaks

## 1. load the cube

* **real data (`use_demo = False`)**: the notebook needs the cube and stdev fits files (and the hdf5 file if `check_hdf5 = True`). if any are missing it stops with an error that lists them.
* **demo (`use_demo = True`)**: builds a fake cube with four things added on purpose: a bad timestep, a patch of nans, a source the model subtraction missed, and a real pulse.

In [ ]:
def check_files_exist():
    """stop with a clear error if the folder or any needed file is missing"""
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(
            "the folder does not exist: " + data_dir + "\n"
            "check obsnum, project_name and data_dir in the settings cell "
            "(is the setonix scratch folder mounted on this computer?)")
    needed_files = [cube_file, stdev_file]
    if check_hdf5:
        needed_files.append(hdf5_file)
    missing_files = []
    for path in needed_files:
        if not os.path.exists(path):
            missing_files.append(path)
    if len(missing_files) > 0:
        message = "cannot find these files:\n"
        for path in missing_files:
            message = message + "    " + path + "\n"
        message = message + "check obsnum and file_prefix in the settings cell, "
        message = message + "or set check_hdf5 = False if you have no hdf5 file, "
        message = message + "or set use_demo = True to see the notebook run on fake data."
        raise FileNotFoundError(message)


def make_demo_cube():
    """fake cube: 10 mjy noise plus four things added on purpose, so every check has something to find"""
    n_timesteps = 70
    size = 300
    random = np.random.default_rng(1)
    cube = random.normal(0, 0.010, (n_timesteps, size, size)).astype("float32")

    # added on purpose 1: one timestep that is three times noisier than the rest
    cube[17] = cube[17] * 3.0

    # added on purpose 2: a patch of nans in one timestep
    cube[5, 200:240, 50:90] = np.nan

    # added on purpose 3: a source the model subtraction missed (same flux at every timestep)
    static_source = 0.08 * make_blob(size, size, 60, 60, 4.0)
    for step in range(n_timesteps):
        cube[step] = cube[step] + static_source

    # added on purpose 4: a real pulse (rises and falls in time, beam-sized in space)
    pulse_position = make_blob(size, size, 220, 210, 4.0)
    for step in range(n_timesteps):
        pulse_height = 0.12 * np.exp(-0.5 * ((step - 35) / 3.0) ** 2)
        cube[step] = cube[step] + pulse_height * pulse_position

    obsnum_number = int(obsnum)
    header = fits.Header()
    header["CTYPE1"] = "RA---SIN"
    header["CTYPE2"] = "DEC--SIN"
    header["CRVAL1"] = 340.5
    header["CRVAL2"] = -38.3
    header["CRPIX1"] = size / 2
    header["CRPIX2"] = size / 2
    header["CDELT1"] = -0.0076
    header["CDELT2"] = 0.0076
    header["CUNIT1"] = "deg"
    header["CUNIT2"] = "deg"
    header["CTYPE3"] = "TIME"
    header["CRVAL3"] = obsnum_number
    header["CDELT3"] = 4
    header["DATE-OBS"] = Time(obsnum_number + 10, format="gps").utc.isot   # first image is 10 s after the obsnum
    stdev_image = np.nanstd(cube, axis=0)
    wcs = WCS(header).celestial
    return cube, stdev_image, header, wcs


if use_demo:
    print("=" * 72)
    print("demo mode: everything below is computed from FAKE data.")
    print("your files are not read. images are saved in " + results_dir)
    print("=" * 72)
    cube, stdev_image, header, wcs = make_demo_cube()
else:
    check_files_exist()
    fits_file = fits.open(cube_file, memmap=True)    # memmap: each timestep is read from disk only when needed
    header = fits_file[0].header
    cube = fits_file[0].data
    if cube.ndim > 3:
        cube = np.squeeze(cube)
    stdev_image = np.squeeze(fits.getdata(stdev_file))
    wcs = WCS(header).celestial
    if cutout is not None:
        y_start, y_end, x_start, x_end = cutout
        cube = cube[:, y_start:y_end, x_start:x_end]
        stdev_image = stdev_image[y_start:y_end, x_start:x_end]
        wcs = wcs.slice((slice(y_start, y_end), slice(x_start, x_end)))

os.makedirs(results_dir, exist_ok=True)

n_timesteps, n_y, n_x = cube.shape
seconds_per_step = float(header.get("CDELT3", 4))

box = min(box_size, n_y, n_x)
box_y_start = (n_y - box) // 2
box_x_start = (n_x - box) // 2


def central_box(image):
    """the central square of an image, used to measure noise away from the edges"""
    return image[box_y_start:box_y_start + box, box_x_start:box_x_start + box]


if use_demo:
    say("observation: " + obsnum + " [demo data, not real]")
    say("data folder: not used in demo mode")
else:
    say("observation: " + obsnum)
    say("data folder: " + data_dir)
say("cube shape (time, y, x) = " + str(cube.shape) + ", seconds per step = " + str(seconds_per_step))
say("noise measured in the central " + str(box) + " x " + str(box) + " pixels")
if cutout is not None:
    say("cutout in use (y_start, y_end, x_start, x_end) = " + str(cutout))

## 2. time axis

`CRVAL3` in the cube is set from the **file name (the obsnum)**, but the first image starts later (the first and last 8 s are skipped). for absolute times use the `DATE-OBS` of each image or the hdf5 `timestamp` dataset, not `CRVAL3`.

In [ ]:
crval3 = header.get("CRVAL3")
say("CTYPE3 = " + str(header.get("CTYPE3")) + ", CRVAL3 = " + str(crval3) + ", CDELT3 = " + str(header.get("CDELT3")) + " s")

if header.get("DATE-OBS") is not None:
    first_image_gps = Time(header["DATE-OBS"], scale="utc").gps
    offset_seconds = first_image_gps - crval3
    say("first image DATE-OBS = " + str(header["DATE-OBS"]) + " (gps " + format(first_image_gps, ".1f") + ")")
    say("CRVAL3 is " + format(offset_seconds, "+.1f") + " s away from the first image")
    if abs(offset_seconds) > 1:
        note_problem("CRVAL3 is " + format(offset_seconds, "+.1f") + " s from the first image. "
                     "use DATE-OBS or the hdf5 timestamps when you need absolute times")
else:
    say("no DATE-OBS in the cube header, cannot compare with CRVAL3")

## 3. noise at each timestep

for every 4 s image, measure the robust rms, the median and the fraction of nans in the central box.

* **healthy:** rms roughly flat in time, median close to 0, no nans. compare the typical rms with wsclean's *"estimated standard deviation of background noise"* in the `.o` log.
* **problem:** a spike or step usually means a bad timestep. check `_summary.txt` for heavy flagging at that time.

the same loop also builds what the next section needs: the sum of all images (for the mean image) and the histogram of pixel values.

In [ ]:
rms_per_step = np.zeros(n_timesteps)
median_per_step = np.zeros(n_timesteps)
nan_fraction_per_step = np.zeros(n_timesteps)

sum_image = np.zeros((n_y, n_x))      # running sum of all timesteps, for the mean image
count_image = np.zeros((n_y, n_x))    # how many timesteps had a real number (not nan) at each pixel

histogram_edges = np.linspace(-8, 8, 81)       # 80 bins from -8 to +8 (units of rms)
histogram_counts = np.zeros(80)

for step in range(n_timesteps):
    image = np.array(cube[step], dtype=float)         # read one timestep
    is_valid = np.isfinite(image)

    sum_image = sum_image + np.where(is_valid, image, 0.0)
    count_image = count_image + is_valid.astype(float)

    centre = central_box(image)
    rms_per_step[step] = robust_rms(centre)
    median_per_step[step] = np.nanmedian(centre)
    nan_fraction_per_step[step] = 1.0 - np.isfinite(centre).mean()

    values_in_rms_units = (centre - median_per_step[step]) / rms_per_step[step]
    values_in_rms_units = values_in_rms_units[np.isfinite(values_in_rms_units)]
    counts_this_step, edges = np.histogram(values_in_rms_units, bins=histogram_edges)
    histogram_counts = histogram_counts + counts_this_step

typical_rms = np.nanmedian(rms_per_step)

# a timestep is "odd" if its noise is far from typical, it has nans, or its median is far from zero
bad_steps = []
for step in range(n_timesteps):
    rms_is_off = rms_per_step[step] > 1.5 * typical_rms or rms_per_step[step] < 0.5 * typical_rms
    has_nans = nan_fraction_per_step[step] > 0.01
    median_is_off = abs(median_per_step[step]) > 0.5 * rms_per_step[step]
    if rms_is_off or has_nans or median_is_off:
        bad_steps.append(step)

jy_to_mjy = 1000.0
steps = np.arange(n_timesteps)
bad_rms_values = []
for step in bad_steps:
    bad_rms_values.append(rms_per_step[step] * jy_to_mjy)

fig, (ax_rms, ax_nan) = plt.subplots(1, 2, figsize=(11, 4))
ax_rms.plot(steps, rms_per_step * jy_to_mjy, "o-", markersize=3, label="each timestep")
ax_rms.axhline(typical_rms * jy_to_mjy, color="black", linestyle=":", label="typical")
ax_rms.plot(bad_steps, bad_rms_values, "ro", label="odd timesteps")
ax_rms.set_xlabel("timestep")
ax_rms.set_ylabel("robust rms (mjy/beam)")
ax_rms.set_title("noise vs time (should be flat)")
ax_rms.legend()
ax_nan.plot(steps, nan_fraction_per_step * 100, "s-", markersize=3, color="green")
ax_nan.set_xlabel("timestep")
ax_nan.set_ylabel("nan fraction in the box (%)")
ax_nan.set_title("nans vs time (should be zero)")
fig.suptitle(make_title("noise at each timestep"))
fig.tight_layout()
save_figure(fig, "01_noise_vs_time.png")

say("typical rms = " + format(typical_rms * jy_to_mjy, ".2f") + " mjy/beam "
    "(smallest " + format(np.nanmin(rms_per_step) * jy_to_mjy, ".2f") + ", "
    "largest " + format(np.nanmax(rms_per_step) * jy_to_mjy, ".2f") + ")")
if len(bad_steps) > 0:
    note_problem(str(len(bad_steps)) + " odd timestep(s): " + str(bad_steps) +
                 " (rms off by more than 50%, more than 1% nans, or median far from zero)")
else:
    say("ok: all timesteps look alike, no nans")

## 4. mean image, stdev image, pixel histogram

* **mean image**: the static sky was subtracted, so this should look like noise with an rms of about *rms / sqrt(number of timesteps)*. a source standing out here was **not removed** by the model subtraction (wrong data column? missing `MODEL_DATA`? phase-centre mismatch?).
* **stdev image**: the "what varies" map. its median should be about the rms of a single timestep (ratio about 1). a **leftover static source** is bright in the mean image but its stdev is just typical. a **real transient** is bright in the stdev image and only weak in the mean (a pulse that is on for a few of the 70 timesteps adds little to the average). bright known sources can also show up in the stdev image because of calibration or ionosphere wobble.
* **histogram** of all pixel values: close to a gaussian. slightly fat tails are normal.

In [ ]:
mean_image = sum_image / np.maximum(count_image, 1)     # max(count, 1) avoids dividing by zero
mean_image[count_image == 0] = np.nan

expected_mean_noise = typical_rms / np.sqrt(n_timesteps)  # what pure noise would give
mean_noise = robust_rms(central_box(mean_image))
typical_stdev = np.nanmedian(central_box(stdev_image))

say("rms of the mean image = " + format(mean_noise * jy_to_mjy, ".2f") + " mjy; "
    "pure noise would give " + format(expected_mean_noise * jy_to_mjy, ".2f") + " mjy "
    "(ratio " + format(mean_noise / expected_mean_noise, ".2f") + ")")
say("typical stdev = " + format(typical_stdev * jy_to_mjy, ".2f") + " mjy; "
    "rms of one timestep = " + format(typical_rms * jy_to_mjy, ".2f") + " mjy "
    "(ratio " + format(typical_stdev / typical_rms, ".2f") + ")")

mean_offset = np.abs(mean_image - np.nanmedian(central_box(mean_image)))
mean_offset = np.nan_to_num(mean_offset)          # nans become 0
strongest_mean_pixel = mean_offset.max() / expected_mean_noise

if mean_noise / expected_mean_noise > 2:
    note_problem("the mean image is much noisier than pure noise: leftover static structure")
if strongest_mean_pixel > 10:
    note_problem("the mean image has a pixel " + format(strongest_mean_pixel, ".0f") + " times the expected noise: "
                 "a source was not removed by the model subtraction")
if typical_stdev / typical_rms < 0.7 or typical_stdev / typical_rms > 1.3:
    note_problem("typical stdev differs by more than 30% from the rms of one timestep")

# --- figure 02: mean image
limit = 5 * expected_mean_noise
fig, ax = plt.subplots(figsize=(7, 6))
picture = ax.imshow(mean_image, vmin=-limit, vmax=limit, cmap="RdBu_r")
fig.colorbar(picture, ax=ax, label="jy/beam (colour range = +-5 x expected noise)")
ax.set_xlabel("x pixel")
ax.set_ylabel("y pixel")
ax.set_title(make_title("mean image over time"))
save_figure(fig, "02_mean_image.png")

# --- figure 03: stdev image with the strongest peaks circled and numbered
peaks = find_top_peaks(stdev_image, how_many=5)
fig, ax = plt.subplots(figsize=(7, 6))
picture = ax.imshow(stdev_image, vmin=0, vmax=np.nanpercentile(stdev_image, 99.5))
fig.colorbar(picture, ax=ax, label="jy/beam")
for number in range(1, 6):
    peak_x, peak_y = peaks[number - 1]
    ax.plot(peak_x, peak_y, "o", markerfacecolor="none", markeredgecolor="white", markersize=16)
    ax.text(peak_x + 10, peak_y + 10, str(number), color="white", fontsize=12)
ax.set_xlabel("x pixel")
ax.set_ylabel("y pixel")
ax.set_title(make_title("stdev through time (top 5 peaks circled)"))
save_figure(fig, "03_stdev_image.png")

# --- figure 04: histogram of all pixel values compared with a gaussian
bin_centres = 0.5 * (histogram_edges[1:] + histogram_edges[:-1])
bin_width = bin_centres[1] - bin_centres[0]
data_density = histogram_counts / histogram_counts.sum() / bin_width
gaussian_density = np.exp(-0.5 * bin_centres ** 2) / np.sqrt(2 * np.pi)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.semilogy(bin_centres, data_density, "o", markersize=4, label="data")
ax.semilogy(bin_centres, gaussian_density, "k-", label="gaussian")
ax.set_ylim(1e-8, 1)
ax.set_xlabel("pixel value / rms of its timestep")
ax.set_ylabel("fraction of pixels per unit")
ax.set_title(make_title("pixel histogram, all timesteps"))
ax.legend()
save_figure(fig, "04_pixel_histogram.png")

# --- table of the peaks
say("")
say("the 5 highest peaks in the stdev image")
say("number     x     y        ra       dec   stdev/typical   mean/(expected noise)")
for number in range(1, 6):
    peak_x, peak_y = peaks[number - 1]
    ra, dec = wcs.pixel_to_world_values(peak_x, peak_y)
    stdev_ratio = stdev_image[peak_y, peak_x] / typical_stdev
    mean_ratio = mean_offset[peak_y, peak_x] / expected_mean_noise
    say(format(number, "6d") + format(peak_x, "6d") + format(peak_y, "6d") +
        format(float(ra), "10.3f") + format(float(dec), "10.3f") +
        format(stdev_ratio, "15.1f") + format(mean_ratio, "20.1f"))
say("read the last two columns together: high stdev with a modest mean = transient-like or variable;")
say("high mean with a typical stdev = leftover static source.")

## 5. light curves

numbers only go so far. this plots the light curve of each of the 5 strongest stdev peaks (blue) next to a typical blank pixel (grey). a pulse shows as a bump over several consecutive timesteps, a bad timestep as a single-sample spike, and a calibration or ionosphere problem as a slow wander.

set `use_sliders = True` in the settings (with `ipywidgets` installed) to look at any pixel you like.

In [ ]:
seconds = np.arange(n_timesteps) * seconds_per_step


def find_typical_pixel(image):
    """the pixel whose value is closest to the median of the image (a blank, boring pixel)"""
    difference = np.abs(image - np.nanmedian(central_box(image)))
    difference[np.isnan(difference)] = np.inf       # never pick a nan pixel
    flat_index = np.argmin(difference)
    y, x = np.unravel_index(flat_index, difference.shape)
    return x, y


def draw_light_curve(ax, x, y, label):
    """draw the light curve of pixel (x, y) on the given axes"""
    light_curve = np.array(cube[:, y, x], dtype=float)
    ax.plot(seconds, light_curve * jy_to_mjy, "o-", markersize=3, label=label)


control_x, control_y = find_typical_pixel(stdev_image)
control_light_curve = np.array(cube[:, control_y, control_x], dtype=float)

fig, axes = plt.subplots(5, 1, figsize=(9, 13), sharex=True)
for number in range(1, 6):
    ax = axes[number - 1]
    peak_x, peak_y = peaks[number - 1]
    ax.plot(seconds, control_light_curve * jy_to_mjy, "-", color="0.65", label="typical pixel (" + str(control_x) + ", " + str(control_y) + ")")
    draw_light_curve(ax, peak_x, peak_y, "peak " + str(number) + " (" + str(peak_x) + ", " + str(peak_y) + ")")
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_ylabel("mjy/beam")
    ax.legend(loc="best")
axes[4].set_xlabel("seconds since the first image")
fig.suptitle(make_title("light curves of the 5 strongest stdev peaks"))
fig.tight_layout()
save_figure(fig, "05_light_curves_top_peaks.png")

if use_sliders:
    try:
        import ipywidgets
    except ImportError:
        ipywidgets = None
    if ipywidgets is None:
        say("ipywidgets is not installed, so there are no sliders")
    else:
        def show_pixel(x, y):
            fig, ax = plt.subplots(figsize=(9, 3.8))
            ax.plot(seconds, control_light_curve * jy_to_mjy, "-", color="0.65", label="typical pixel")
            draw_light_curve(ax, x, y, "pixel (" + str(x) + ", " + str(y) + ")")
            ax.set_xlabel("seconds since the first image")
            ax.set_ylabel("mjy/beam")
            ax.legend()
            plt.show()

        x_slider = ipywidgets.IntSlider(min=0, max=n_x - 1, value=peaks[0][0], continuous_update=False)
        y_slider = ipywidgets.IntSlider(min=0, max=n_y - 1, value=peaks[0][1], continuous_update=False)
        ipywidgets.interact(show_pixel, x=x_slider, y=y_slider)

## 6. toy injection test: would a pulse of this strength be visible?

a good check of any search is to **add a fake pulse and see whether it comes out**. here a gaussian pulse (a width in time and a beam-sized spot in space) is added to 25 places in a blank region of the cube. then a simple **matched filter in time** is applied: each pixel's light curve is smoothed with a gaussian of the same width, and the peak signal-to-noise is read off.

this is only an illustration of the idea (similar in spirit to the `tcg` filter in `tfilter.tmpl`). it is **not** the real `DoFilter.py`, whose settings and noise handling differ. to test the real pipeline, write the injected cube to an hdf5 file and run `obs_tfilter.sh` on it.

the two pulse settings below are guesses: `spatial_fwhm_pixels` should be the restored beam from the wsclean log divided by your pixel size.

In [ ]:
pulse_fwhm_seconds = 20.0       # width of the fake pulse in time
spatial_fwhm_pixels = 4.0       # width of the fake source in space
half_size = 60                  # the test region is 2 x half_size pixels wide, in the middle of the field

# cut out the test region (nans become 0)
centre_y = n_y // 2
centre_x = n_x // 2
region = np.array(cube[:, centre_y - half_size:centre_y + half_size, centre_x - half_size:centre_x + half_size], dtype=float)
region = np.nan_to_num(region)
region_size = 2 * half_size

# 25 injection sites on a grid, 24 pixels apart so the fake sources do not overlap
sites = []
for y in range(12, region_size - 6, 24):
    for x in range(12, region_size - 6, 24):
        sites.append((y, x))

# shape in space: one blob per site added together. shape in time: one gaussian pulse in the middle of the observation.
spatial_profile = np.zeros((region_size, region_size))
for y, x in sites:
    spatial_profile = spatial_profile + make_blob(region_size, region_size, y, x, spatial_fwhm_pixels)

sigma_steps = pulse_fwhm_seconds / 2.355 / seconds_per_step      # pulse width in timesteps
time_profile = np.exp(-0.5 * ((np.arange(n_timesteps) - n_timesteps / 2) / sigma_steps) ** 2)
time_profile_3d = time_profile.reshape(n_timesteps, 1, 1)
spatial_profile_3d = spatial_profile.reshape(1, region_size, region_size)


def matched_filter_snr(data):
    """smooth every light curve with a gaussian the width of the pulse, then take its peak in time,
    in units of the noise of the smoothed data.
    zero padding (mode constant) is used at the ends: repeating the edge samples would inflate the noise there
    and create false detections in the first and last few timesteps."""
    smoothed = gaussian_filter1d(data, sigma=sigma_steps, axis=0, mode="constant")
    peak_in_time = smoothed.max(axis=0)
    return peak_in_time / robust_rms(smoothed)


noise_per_step = robust_rms(region[n_timesteps // 2])
filter_gain = np.sqrt(0.5) * np.sqrt(2 * np.sqrt(np.pi) * sigma_steps)    # theory for a gaussian pulse and an equal-width filter

say("")
say("injection test")
say("noise in the test region = " + format(noise_per_step * jy_to_mjy, ".1f") + " mjy/beam per timestep; "
    "pulse width = " + str(pulse_fwhm_seconds) + " s = " + format(pulse_fwhm_seconds / seconds_per_step, ".1f") + " timesteps")
say("expected filter gain for this width = " + format(filter_gain, ".1f") + " x (peak/rms); single pixels scatter by about +-1 around it")
say("")
say("peak height (x rms)   single-timestep s/n   matched-filter s/n: median (smallest - largest) over " + str(len(sites)) + " sites")

snr_map_for_plot = None
for peak_height in [2, 3, 4, 6, 10]:
    fake_pulses = (peak_height * noise_per_step) * time_profile_3d * spatial_profile_3d
    snr_map = matched_filter_snr(region + fake_pulses)
    snr_at_sites = []
    for y, x in sites:
        snr_at_sites.append(snr_map[y, x])
    say(format(peak_height, "14d") + format(float(peak_height), "22.1f") + format(np.median(snr_at_sites), "24.1f") +
        "   (" + format(min(snr_at_sites), ".1f") + " - " + format(max(snr_at_sites), ".1f") + ")")
    if peak_height == 4:
        snr_map_for_plot = snr_map

# false detections: pixels above 5 sigma when nothing was injected
snr_without_injection = matched_filter_snr(region)
say("")
say("with no injection, pixels above 5 sigma in this region: " + str((snr_without_injection > 5).sum()) +
    " (highest " + format(snr_without_injection.max(), ".1f") + ")")

region_cleaned = region.copy()
for step in bad_steps:
    region_cleaned[step] = 0                      # blank the odd timesteps found in section 3
snr_after_blanking = matched_filter_snr(region_cleaned)
say("after blanking the odd timesteps " + str(bad_steps) + ": " + str((snr_after_blanking > 5).sum()) +
    " pixels above 5 sigma (highest " + format(snr_after_blanking.max(), ".1f") + ")")
say("a bad timestep creates false detections across the whole image, so flag it before filtering.")

# --- figure 06: what the filter does to one light curve, and the s/n map
site_y, site_x = sites[len(sites) // 2]
one_pulse = (region + (4 * noise_per_step) * time_profile_3d * spatial_profile_3d)[:, site_y, site_x]
one_pulse_smoothed = gaussian_filter1d(one_pulse, sigma=sigma_steps, mode="constant")

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(12, 4.5))
ax_left.plot(seconds, one_pulse * jy_to_mjy, "o-", markersize=3, label="one injected pixel (peak = 4 x rms)")
ax_left.plot(seconds, one_pulse_smoothed * jy_to_mjy, "r-", label="after the matched filter")
ax_left.set_xlabel("seconds")
ax_left.set_ylabel("mjy/beam")
ax_left.set_title("what the filter does to one light curve")
ax_left.legend()
picture = ax_right.imshow(snr_map_for_plot, vmin=0, vmax=max(8, snr_map_for_plot.max()))
fig.colorbar(picture, ax=ax_right, label="signal-to-noise")
ax_right.set_title("matched-filter s/n map, " + str(len(sites)) + " injections of height 4 x rms")
fig.suptitle(make_title("toy injection test"))
fig.tight_layout()
save_figure(fig, "06_injection_test.png")

## 7. hdf5 cross-check

checks that the hdf5 file made by `make_imstack.py` has the same number of timesteps as the cube, that its timestamps are one step apart, and that its images agree with the cube.

In [ ]:
if use_demo:
    say("hdf5 check skipped: demo mode has no hdf5 file")
elif not check_hdf5:
    say("hdf5 check skipped: check_hdf5 is False")
else:
    import h5py
    with h5py.File(hdf5_file, "r") as f:
        say("hdf5 datasets: " + str(list(f.keys())))

        # timestamps: convert each one to gps seconds
        timestamps_gps = []
        for text in f["timestamp"][:]:
            if isinstance(text, bytes):
                text = text.decode()
            timestamps_gps.append(Time(text, scale="utc").gps)
        timestamps_gps = np.array(timestamps_gps)
        step_sizes = np.diff(timestamps_gps)
        say("timestamps: " + str(len(timestamps_gps)) + " of them, spacing " +
            format(step_sizes.min(), ".1f") + " to " + format(step_sizes.max(), ".1f") + " s, first = gps " +
            format(timestamps_gps[0], ".1f"))
        if len(timestamps_gps) != n_timesteps:
            note_problem("the hdf5 file has " + str(len(timestamps_gps)) + " timesteps but the cube has " + str(n_timesteps))
        if step_sizes.max() - step_sizes.min() > 0.5:
            note_problem("the hdf5 timestamps are not evenly spaced")

        # compare the first, middle and last image with the cube (hdf5 stores float16, so tiny differences are expected)
        for step in [0, n_timesteps // 2, n_timesteps - 1]:
            hdf5_image = np.array(f["image"][0, :, :, 0, step], dtype=float)
            if cutout is not None:
                hdf5_image = hdf5_image[y_start:y_end, x_start:x_end]
            cube_image = np.array(cube[step], dtype=float)
            biggest_difference = np.nanmax(np.abs(hdf5_image - cube_image))
            say("timestep " + str(step) + ": biggest difference between hdf5 and cube = " +
                format(biggest_difference / typical_rms, ".4f") + " x rms")
            if biggest_difference / typical_rms > 0.05:
                note_problem("the hdf5 image and the cube disagree at timestep " + str(step))

## 8. summary and saved files

In [ ]:
say("")
if len(problems) > 0:
    say("things to look at:")
    for problem in problems:
        say("  - " + problem)
else:
    say("no problems flagged by these checks.")
if use_demo:
    say("")
    say("(demo mode: these are the problems added on purpose, plus the CRVAL3 offset that every real cube has.)")

report_path = os.path.join(results_dir, "report.txt")
with open(report_path, "w") as report_file:
    for line in report_lines:
        report_file.write(line + "\n")

print("")
print("files saved in " + os.path.abspath(results_dir) + ":")
for file_name in sorted(os.listdir(results_dir)):
    print("   " + file_name)

### what these checks do not tell you

they show that the cube is sane: noise-like, model subtraction worked, timesteps good. they do **not** prove that real transients would be found. for that, inject into the actual pipeline (the idea in section 6) and compare against known variable sources or pulsars in the field. a source in the stdev image is also not automatically a transient: bright known sources vary because of calibration and the ionosphere.